# Transfer Learning for Bioacoustics

In this notebook, we walk through a simple but very useful workflow in bioacoustics: using a pretrained model to turn audio recordings into embeddings, and then training lightweight classifiers on top of those embeddings.

This setup is especially helpful when labeled data is limited. Instead of training a large model from scratch, we let a pretrained model do the heavy lifting and then build small downstream models for the task we care about.

In this tutorial we will:

- prepare a small train/test split from raw recordings
- generate embeddings with a pretrained model
- visualize the embedding space with dimensionality reduction
- train a linear probe on top of the embeddings
- evaluate the classifier on a held-out test set
- run a small few-shot experiment to see how performance changes as the number of training recordings increases

The goal here is not only to get a result, but to understand the workflow and build some intuition for transfer learning in practice.


## 1. Environment setup

We begin by installing the packages needed for the tutorial. In Colab this is necessary because the runtime is temporary, so each new session starts from a clean slate.

Note: Please restart the session after installing the packages.


In [ ]:
!pip install uv
!uv pip install "bacpipe==1.3.0.dev4" "numpy==1.26.4" --refresh-package bacpipe
!pip install jupyter_bokeh


## 2. Imports

Here we import the main libraries used throughout the notebook.

`bacpipe` handles the embedding pipeline, while the rest of the imports cover file handling, plotting, audio utilities, and the small PyTorch model used later for classification.


In [ ]:
import bacpipe
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import random
import shutil
import librosa
import soundfile as sf
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import json
import matplotlib.pyplot as plt
from google.colab import drive
%matplotlib inline


## 3. Accessing the dataset

For this workshop version, the data lives on Google Drive, so we mount the drive here. If you are running the notebook locally, you can skip this step and simply point the dataset paths to a local directory.


In [ ]:
# Mount drive
drive.mount('/content/drive')


# User configuration
DATASET_ROOT = Path("/content/drive/MyDrive/bioacoustics_workshop")
MODEL_ROOT = Path("/content/drive/MyDrive/bioacoustics_workshop_models")
RESULTS_ROOT = Path("/content/bacpipe_results")

# Example dataset choices
DATASET_NAME = "yellowhammer_dialects_subset"
DATASET_DIR = DATASET_ROOT / DATASET_NAME

## 4. List of available pre-trained models

Displays a list of all models you can choose from.


In [ ]:
def print_output(var):
    print(f'\n{var=}\n')

# Print supported models
print_output(bacpipe.supported_models)


## 5. Creating a simple train/test split from the audio files

Before we train anything, we first create a train/test split at the level of audio recordings. The code below takes the recordings inside each class folder, shuffles them, and copies them into new `train/` and `test/` directories.

This is a practical step for the workshop: it gives us a clear directory structure and makes the later embedding and evaluation steps easier to follow.


In [ ]:
# Settings
src_root = DATASET_DIR   # species folders here
test_ratio = 0.2                                   # e.g., 20% test
seed = 42
TRAINSET_NAME=f"{src_root.name}_train_test"
out_root = DATASET_ROOT / TRAINSET_NAME
train_root = out_root / "train"
test_root = out_root / "test"

audio_exts = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac"}

random.seed(seed)

train_root.mkdir(parents=True, exist_ok=True)
test_root.mkdir(parents=True, exist_ok=True)

for species_dir in sorted([d for d in src_root.iterdir() if d.is_dir()]):
    files = [p for p in species_dir.iterdir() if p.is_file() and p.suffix.lower() in audio_exts]

    if not files:
        print(f"Skipping {species_dir.name} (no audio files)")
        continue

    random.shuffle(files)

    n_total = len(files)
    n_test = int(n_total * test_ratio)
    n_train = n_total - n_test

    train_files = files[:n_train]
    test_files = files[n_train:]

    # create species dirs
    train_species_dir = train_root / species_dir.name
    test_species_dir = test_root / species_dir.name

    train_species_dir.mkdir(parents=True, exist_ok=True)
    test_species_dir.mkdir(parents=True, exist_ok=True)

    # copy files
    for f in train_files:
        shutil.copy2(f, train_species_dir / f.name)

    for f in test_files:
        shutil.copy2(f, test_species_dir / f.name)

    print(f"{species_dir.name}: {n_train} train / {n_test} test")

print(f"\nDone. Dataset created at:\n{out_root}")


## 6. Generating embeddings with a pretrained model

Now that the dataset is arranged into train and test folders, we can feed the recordings into a pretrained model and extract embeddings.

This is the heart of transfer learning here: we reuse a model that has already learned useful acoustic representations and treat its output embeddings as features for our downstream task.

We first make sure the model checkpoint is available, and then we run the embedding generation step. After that, we also ask `bacpipe` to create a dimensionality-reduced version of the embeddings for visualization.


In [ ]:
bacpipe.settings.model_base_path = '/content/bacpipe/model_checkpoints'
bacpipe.settings.device = 'cpu'
bacpipe.ensure_models_exist(
    bacpipe.settings.model_base_path,
     ['birdnet']
    )

# Generate model embeddings
loader_obj = bacpipe.generate_embeddings(
    model_name='birdnet',
    audio_dir=out_root
)

# Generate dimensionality reduced embeddings
loader_obj = bacpipe.generate_embeddings(
    model_name='birdnet',
    audio_dir=out_root,
    dim_reduction_model='t_sne'
)


## 7. Inspecting the embedding space

Once the embeddings and their dimensionality-reduced version have been saved, it is useful to visualize them. This does not tell us everything, but it gives a quick qualitative sense of whether the classes are already separating in the learned feature space.

In the next cell we load the reduced coordinates from JSON, recover the class labels and split information, and then visualize only the test recordings.

Note that you will need to provide the path to the generated JSON file. This file is created automatically by `bacpipe` inside the bacpipe_results directory. The easiest way is to navigate to the corresponding folder in `bacpipe_results` and copy the full path of the JSON file. A sample path is already included in the script to illustrate the expected structure. Replace it with the path that matches your own run.


In [ ]:

# Load JSON

json_path = "/content/bacpipe_results/yellowhammer_dialects_subset_train_test/dim_reduced_embeddings/2026-03-19_11-38___t_sne-yellowhammer_dialects_subset_train_test-birdnet/yellowhammer_dialects_subset_train_test_birdnet.json"

with open(json_path, "r") as f:
    data = json.load(f)

x = np.array(data["x"])
y = np.array(data["y"])

audio_files = data["metadata"]["audio_files"]
nr_per_file = data["metadata"]["nr_embeds_per_file"]


# Build labels per embedding
labels = []
split_mask = []  # True if test

idx = 0
for file_path, n in zip(audio_files, nr_per_file):

    # extract split and class (example: test/B/...)

    parts = file_path.split("/")
    split = parts[0]   # "train" or "test"
    cls = parts[1]     # "B" or "X"

    for _ in range(n):
        labels.append(cls)
        split_mask.append(split == "test")
        idx += 1

labels = np.array(labels)
split_mask = np.array(split_mask)


# Filter TEST only
x_test = x[split_mask]
y_test = y[split_mask]
labels_test = labels[split_mask]


# Plot
plt.figure(figsize=(8, 6))

for cls in np.unique(labels_test):
    mask = labels_test == cls
    plt.scatter(
        x_test[mask],
        y_test[mask],
        label=cls,
        alpha=0.7,
        s=30
    )

plt.legend(title="Class")
plt.title("t-SNE plot")
plt.xlabel("dim-1")
plt.ylabel("dim-2")
plt.grid(True)
plt.show()


## 8. Training a linear probe on the embeddings

The next step is to ask a simple question: how informative are these embeddings for the classification problem?

A common way to test that is to train a linear probe, which is just a single linear layer on top of frozen embeddings. If a linear classifier works well, that usually means the embeddings already encode the class information in a very usable way.

The data loading function below also handles the fact that some recordings may contain multiple chunk-level embeddings. In that case, each chunk is treated as a separate example and inherits the class label of the parent recording.

Note that you will need to provide the path to the embeddings directory. The easiest way is to navigate to the corresponding folder in `bacpipe_results` and copy the full path. A sample path is already included in the script to illustrate the expected structure. Replace it with the path that matches your own run.


In [ ]:

# Config
EMBED_ROOT = Path("/content/bacpipe_results/yellowhammer_dialects_subset_train_test/embeddings/2026-03-19_11-33___birdnet-yellowhammer_dialects_subset_train_test")
TRAIN_DIR = EMBED_ROOT / "train"
BATCH_SIZE = 256
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# data loading

def load_split(split_dir):
    X, y = [], []

    species_dirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])
    if not species_dirs:
        raise ValueError(f"No class folders found in {split_dir}")

    for species_dir in species_dirs:
        species = species_dir.name
        files = sorted(species_dir.glob("*.npy"))

        if not files:
            print(f"Warning: no .npy files found in {species_dir}")
            continue

        for f in files:
            emb = np.load(f)
            emb = np.asarray(emb, dtype=np.float32)

            # Case 1: single embedding vector, shape (emb_dim,)
            if emb.ndim == 1:
                X.append(emb)
                y.append(species)

            # Case 2: chunked embeddings, shape (n_chunks, emb_dim)
            elif emb.ndim == 2:
                for chunk_emb in emb:
                    X.append(chunk_emb)
                    y.append(species)

            # Optional: handle weird shapes explicitly
            else:
                raise ValueError(
                    f"Expected 1D or 2D embedding in {f}, got shape {emb.shape}"
                )

    if not X:
        raise ValueError(f"No embeddings loaded from {split_dir}")

    X = np.stack(X).astype(np.float32)
    y = np.array(y)
    return X, y

X_train, y_train_text = load_split(TRAIN_DIR)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_text)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)


# Linear probe

class LinearProbe(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)

    def forward(self, x):
        return self.fc(x)

model = LinearProbe(
    in_dim=X_train.shape[1],
    num_classes=len(label_encoder.classes_)
).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)


# Training loop

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for xb, yb in train_loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | loss={epoch_loss:.4f} | acc={epoch_acc:.4f}")

print("\nTraining complete.")
print(f"Embedding dim: {X_train.shape[1]}")
print(f"Number of classes: {len(label_encoder.classes_)}")
print(f"Number of training examples: {len(X_train)}")


## 9. Evaluating on the held-out test split

After training the probe on the training embeddings, we evaluate it on the embeddings from the test recordings. We report not only accuracy, but also precision, recall, F1-score, and a full per-class report.

This gives us a fuller picture of performance than accuracy alone.


In [ ]:
TEST_DIR = EMBED_ROOT / "test"

# Load test data
X_test, y_test_text = load_split(TEST_DIR)
y_test = label_encoder.transform(y_test_text)

X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)

# Inference
model.eval()
with torch.no_grad():
    logits = model(X_test_t)
    y_pred = logits.argmax(dim=1).cpu().numpy()

# Metrics
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Test accuracy         : {acc:.4f}")
print(f"F1-score              : {f1:.4f}")
print(f"Precision             : {precision:.4f}")
print(f"Recall                : {recall:.4f}")

print("\nPer-class report:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))


## 10. A few-shot learning experiment

Finally, we move to a more realistic low-data scenario.

Instead of relying on the fixed train/test split from above, we pool together the available embeddings for each class and repeatedly sample a small number of recordings per class for training. We then evaluate on the rest.

This lets us answer a very practical question: how well do these embeddings support classification when only a handful of labeled recordings are available?

The experiment is repeated multiple times for each shot value so that the curve is less sensitive to one lucky or unlucky split.




In [ ]:

# Config
BATCH_SIZE = 256
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SHOTS = [4, 8, 16, 32, 64, 128]          # recordings per class
N_REPEATS = 10                  # repeat random split this many times
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# Helpers
def list_class_files(embed_root):
    """
    Returns:
        class_to_files: dict[class_name] -> list of .npy file paths

    Supports either:
      1) embed_root/class_name/*.npy
      2) embed_root/train/class_name/*.npy and embed_root/test/class_name/*.npy

    In case (2), train and test files are lumped together per class.
    """
    train_dir = embed_root / "train"
    test_dir = embed_root / "test"

    # Case 1: already flat class structure
    flat_class_dirs = sorted([d for d in embed_root.iterdir() if d.is_dir() and d.name not in {"train", "test"}])
    if len(flat_class_dirs) == 2:
        class_to_files = {}
        for class_dir in flat_class_dirs:
            files = sorted(class_dir.glob("*.npy"))
            if len(files) == 0:
                raise ValueError(f"No .npy files found in {class_dir}")
            class_to_files[class_dir.name] = files
        return class_to_files

    # Case 2: split structure with train/ and test/
    if train_dir.is_dir() and test_dir.is_dir():
        train_class_dirs = sorted([d for d in train_dir.iterdir() if d.is_dir()])
        test_class_dirs = sorted([d for d in test_dir.iterdir() if d.is_dir()])

        train_class_names = {d.name for d in train_class_dirs}
        test_class_names = {d.name for d in test_class_dirs}

        if train_class_names != test_class_names:
            raise ValueError(
                f"Train/test class mismatch:\n"
                f"train classes = {sorted(train_class_names)}\n"
                f"test classes  = {sorted(test_class_names)}"
            )

        if len(train_class_names) != 2:
            raise ValueError(
                f"Expected exactly 2 classes across train/test in {embed_root}, "
                f"found {len(train_class_names)}"
            )

        class_to_files = {}
        for class_name in sorted(train_class_names):
            train_files = sorted((train_dir / class_name).glob("*.npy"))
            test_files = sorted((test_dir / class_name).glob("*.npy"))
            files = train_files + test_files

            if len(files) == 0:
                raise ValueError(f"No .npy files found for class {class_name}")

            class_to_files[class_name] = files

        return class_to_files

    raise ValueError(
        f"Could not parse class structure in {embed_root}. "
        f"Expected either class folders directly under root, or train/test subfolders."
    )


def load_files(file_list, class_name):
    """
    Loads a list of embedding files for one class.
    Each file can be:
      - 1D: (emb_dim,)
      - 2D: (n_chunks, emb_dim)

    Returns:
        X: [n_examples, emb_dim]
        y: [n_examples] as class_name strings
    """
    X, y = [], []

    for f in file_list:
        emb = np.load(f)
        emb = np.asarray(emb, dtype=np.float32)

        if emb.ndim == 1:
            X.append(emb)
            y.append(class_name)

        elif emb.ndim == 2:
            for chunk_emb in emb:
                X.append(chunk_emb)
                y.append(class_name)

        else:
            raise ValueError(f"Expected 1D or 2D embedding in {f}, got shape {emb.shape}")

    return X, y


def build_dataset(files_by_class):
    """
    files_by_class: dict[class_name] -> list[file paths]

    Returns:
        X: np.ndarray [n_examples, emb_dim]
        y: np.ndarray [n_examples] of class strings
    """
    X_all, y_all = [], []

    for class_name, files in files_by_class.items():
        X, y = load_files(files, class_name)
        X_all.extend(X)
        y_all.extend(y)

    if len(X_all) == 0:
        raise ValueError("No embeddings loaded.")

    X_all = np.stack(X_all).astype(np.float32)
    y_all = np.array(y_all)
    return X_all, y_all


class LinearProbe(nn.Module):
    def __init__(self, in_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(in_dim, num_classes)

    def forward(self, x):
        return self.fc(x)


def train_and_eval(X_train, y_train, X_test, y_test, label_encoder):
    """
    Train linear probe and return test accuracy.
    """
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(label_encoder.transform(y_train), dtype=torch.long)

    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(DEVICE)
    y_test_num = label_encoder.transform(y_test)

    train_ds = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

    model = LinearProbe(
        in_dim=X_train.shape[1],
        num_classes=len(label_encoder.classes_)
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    for epoch in range(EPOCHS):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(X_test_t)
        y_pred = logits.argmax(dim=1).cpu().numpy()

    acc = accuracy_score(y_test_num, y_pred)
    return acc



# Main few-shot experiment
class_to_files = list_class_files(EMBED_ROOT)
class_names = sorted(class_to_files.keys())

print("Classes found:", class_names)
for c in class_names:
    print(f"{c}: {len(class_to_files[c])} recordings")

# Label encoder fixed once from class names
label_encoder = LabelEncoder()
label_encoder.fit(class_names)

results = {shot: [] for shot in SHOTS}

for shot in SHOTS:
    print(f"\n=== {shot}-shot per class ===")

    # Make sure each class has enough recordings
    for c in class_names:
        if len(class_to_files[c]) <= shot:
            raise ValueError(
                f"Class '{c}' has only {len(class_to_files[c])} recordings, "
                f"cannot take {shot} for training and still have test data."
            )

    for repeat in range(N_REPEATS):
        train_files_by_class = {}
        test_files_by_class = {}

        for c in class_names:
            files = class_to_files[c].copy()
            random.shuffle(files)

            train_files = files[:shot]
            test_files = files[shot:]

            train_files_by_class[c] = train_files
            test_files_by_class[c] = test_files

        X_train, y_train = build_dataset(train_files_by_class)
        X_test, y_test = build_dataset(test_files_by_class)

        acc = train_and_eval(X_train, y_train, X_test, y_test, label_encoder)
        results[shot].append(acc)

        print(
            f"Repeat {repeat+1:02d}/{N_REPEATS} | "
            f"train_examples={len(X_train):4d} | "
            f"test_examples={len(X_test):4d} | "
            f"acc={acc:.4f}"
        )


# Summarize
shots = []
mean_accs = []
std_accs = []

print("\n=== Summary ===")
for shot in SHOTS:
    vals = np.array(results[shot], dtype=np.float32)
    mean_acc = vals.mean()
    std_acc = vals.std()

    shots.append(shot)
    mean_accs.append(mean_acc)
    std_accs.append(std_acc)

    print(f"{shot:2d}-shot | acc = {mean_acc:.4f} ± {std_acc:.4f}")


# Plot
plt.figure(figsize=(7, 5))
plt.errorbar(shots, mean_accs, yerr=std_accs, marker='o', capsize=4)
plt.xticks(shots)
plt.xlabel("Number of training recordings per class")
plt.ylabel("Accuracy")
plt.title("Few-shot performance curve")
plt.grid(True)
plt.show()


## Wrap-up

By this point, you have gone through a complete transfer-learning workflow for a small bioacoustics classification task:

- preparing a dataset split
- extracting embeddings with a pretrained model
- visualizing the embedding space
- training and evaluating a linear probe
- testing how the setup behaves in a low-data, few-shot setting

A nice next step for participants is to compare different embedding models, try a different dataset, or change the number of shots and see how the curve moves.
